In [1]:
%pip install requests bs4 tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import random
import time
import urllib3
import requests
import pandas as pd
import pickle

from datetime import datetime
from bs4 import BeautifulSoup
from tqdm import tqdm

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

urllib3.disable_warnings(
    urllib3.exceptions.InsecureRequestWarning
)

In [ ]:
INPUT_FILE = "plants_with_taxonomy.csv"
OUTPUT_FILE = "wfo_descriptions.csv"
FAILED_FILE = "failed_ipni_ids.csv"

BATCH_SIZE = 100
CHECKPOINT_EVERY = 100
SLEEP_TIME = random.uniform(1, 3)
CHECKPOINT_FILE = "header_checkpoint.pkl"

In [ ]:
## TODO: Merge below two blocks together

In [ ]:
INPUT_FILE = "wfo_descriptions.csv"
OUTPUT_FILE = "wfo_descriptions_with_headers.csv"

retry_strategy = Retry(
    total=3,
    connect=3,
    read=3,
    backoff_factor=2,
    status_forcelist=[
        429,
        500,
        502,
        503,
        504
    ],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(
    max_retries=retry_strategy
)

session = requests.Session()

session.mount(
    "http://",
    adapter
)

session.mount(
    "https://",
    adapter
)

session.headers.update({
    "User-Agent": (
        "Mozilla/2.21 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/431.42 "
        "(KHTML, like Gecko) "
        "Chrome/130.0 Safari/525.31"
    )
})

def log(message):
    timestamp = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )
    print(f"[{timestamp}] {message}")


def save_checkpoint(
    header_lookup,
    processed_ipni_ids
):
    with open(
        CHECKPOINT_FILE,
        "wb"
    ) as f:
        pickle.dump(
            {
                "header_lookup": header_lookup,
                "processed_ipni_ids": processed_ipni_ids
            },
            f
        )
    
    log(
        f"CHECKPOINT | "
        f"{len(processed_ipni_ids):,} plants saved"
    )


def load_checkpoint():

    if not os.path.exists(
        CHECKPOINT_FILE
    ):
        return {}, set()

    with open(
        CHECKPOINT_FILE,
        "rb"
    ) as f:
        checkpoint = pickle.load(f)

    log(
        f"RESUME | "
        f"{len(checkpoint['processed_ipni_ids']):,} "
        f"plants already completed"
    )
    
    log(
        f"Checkpoint contains "
        f"{len(checkpoint['header_lookup']):,} "
        f"description headers"
    )

    log(
        f"Checkpoint contains "
        f"{len(checkpoint['processed_ipni_ids']):,} "
        f"completed plants"
    )

    return (
        checkpoint["header_lookup"],
        checkpoint["processed_ipni_ids"]
    )

df = pd.read_csv(INPUT_FILE)

log(
    f"Loaded {len(df):,} rows"
)

# --------------------------------------------------
# Only consider plants that actually have descriptions
# --------------------------------------------------

description_rows = df.loc[
    df["description_id"].notna()
].copy()

# Map:
# ipni_id -> set(description_ids present in CSV)
plant_description_ids = (
    description_rows
    .assign(
        description_id=lambda x:
        x["description_id"].astype(int)
    )
    .groupby("ipni_id")["description_id"]
    .apply(set)
    .to_dict()
)

unique_ipni_ids = [
    str(ipni_id)
    for ipni_id in plant_description_ids.keys()
]

log(
    f"Found {len(unique_ipni_ids):,} plants "
    f"with descriptions"
)

all_plants = (
    df["ipni_id"]
    .astype(str)
    .nunique()
)

log(
    f"Plants in CSV: "
    f"{all_plants:,}"
)

log(
    f"Plants with descriptions: "
    f"{len(unique_ipni_ids):,}"
)

log(
    f"Plants skipped: "
    f"{all_plants - len(unique_ipni_ids):,}"
)

(
    header_lookup,
    processed_ipni_ids
) = load_checkpoint()

remaining_ids = [
    ipni_id
    for ipni_id in unique_ipni_ids
    if ipni_id not in processed_ipni_ids
]

log(
    f"{len(remaining_ids):,} plants "
    f"remaining"
)

processed_since_save = 0

consecutive_block_errors = 0

MAX_CONSECUTIVE_BLOCK_ERRORS = 5

BLOCK_INDICATORS = [
    "too many 500 error responses",
    "Max retries exceeded",
    "ResponseError",
    "SSLEOFError",
    "429",
    "Connection reset",
    "RemoteDisconnected",
]

for ipni_id in tqdm(
    remaining_ids
):

    required_desc_ids = (
        plant_description_ids.get(ipni_id)
    )

    # Should never happen, but be safe
    if not required_desc_ids:

        processed_ipni_ids.add(
            ipni_id
        )

        continue

    url = (
        f"https://www.worldfloraonline.org/"
        f"ipni/{ipni_id}"
    )

    try:

        response = session.get(
            url,
            timeout=30,
            verify=False
        )

        response.raise_for_status()

        # Successful request -> clear block counter
        consecutive_block_errors = 0

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        general = soup.find(
            "div",
            class_="tab-pane active"
        )

        if general:

            descriptions = general.find_all(
                "div",
                class_="description-with-citations"
            )

            for desc_idx, desc in enumerate(
                descriptions
            ):

                # Only keep description IDs
                # that actually exist in the CSV
                if desc_idx not in required_desc_ids:
                    continue

                header = None

                section_div = desc.parent

                h2 = section_div.find(
                    "h2"
                )

                if h2:

                    span = h2.find(
                        "span"
                    )

                    if span:

                        header = (
                            span.get_text(
                                strip=True
                            )
                        )

                    else:

                        header = (
                            h2.get_text(
                                " ",
                                strip=True
                            )
                        )

                if not header:

                    header = section_div.get(
                        "id"
                    )

                header_lookup[
                    (
                        ipni_id,
                        desc_idx
                    )
                ] = header

        processed_ipni_ids.add(
            ipni_id
        )

        processed_since_save += 1

        if (
            processed_since_save
            >= CHECKPOINT_EVERY
        ):

            save_checkpoint(
                header_lookup,
                processed_ipni_ids
            )

            processed_since_save = 0

        time.sleep(
            random.uniform(5, 10)
        )

    except Exception as e:

        error_msg = str(e)

        log(
            f"ERROR | "
            f"{ipni_id} | {error_msg}"
        )

        is_block_error = any(
            indicator in error_msg
            for indicator in BLOCK_INDICATORS
        )

        if is_block_error:

            consecutive_block_errors += 1

            log(
                f"BLOCK WARNING | "
                f"{consecutive_block_errors}/"
                f"{MAX_CONSECUTIVE_BLOCK_ERRORS}"
            )

            if (
                consecutive_block_errors
                >= MAX_CONSECUTIVE_BLOCK_ERRORS
            ):

                log(
                    "LIKELY RATE LIMITED OR BLOCKED. "
                    "Saving checkpoint and exiting."
                )

                save_checkpoint(
                    header_lookup,
                    processed_ipni_ids
                )

                raise SystemExit(
                    "Stopped due to repeated "
                    "connection/block errors."
                )

            # Give WFO a chance to calm down
            time.sleep(60)

        else:

            # Normal parsing/data issue
            consecutive_block_errors = 0

        continue

# Final checkpoint
save_checkpoint(
    header_lookup,
    processed_ipni_ids
)

log(
    f"Extracted "
    f"{len(header_lookup):,} "
    f"headers"
)

df["description_header"] = df.apply(
    lambda row:
        header_lookup.get(
            (
                str(row["ipni_id"]),
                int(row["description_id"])
            )
        )
        if pd.notna(
            row["description_id"]
        )
        else None,
    axis=1
)

df.to_csv(
    OUTPUT_FILE,
    index=False
)

log(
    f"Saved: {OUTPUT_FILE}"
)

matched = (
    df["description_header"]
    .notna()
    .sum()
)

log(
    f"Matched "
    f"{matched:,} / {len(df):,} rows"
)

[2026-10-20 14:56:35] Loaded 23,489 rows
[2026-10-20 14:56:35] Found 5,800 plants with descriptions
[2026-10-20 14:56:35] Plants in CSV: 11,790
[2026-10-20 14:56:35] Plants with descriptions: 5,800
[2026-10-20 14:56:35] Plants skipped: 5,990
[2026-10-20 14:56:35] RESUME | 4,138 plants already completed
[2026-10-20 14:56:35] Checkpoint contains 12,027 description headers
[2026-10-20 14:56:35] Checkpoint contains 4,138 completed plants
[2026-10-20 14:56:35] 1,815 plants remaining


  5%|▌         | 99/1815 [15:46<4:43:18,  9.91s/it]

[2026-10-20 15:12:24] CHECKPOINT | 4,238 plants saved


 11%|█         | 199/1815 [31:20<4:24:50,  9.83s/it]

[2026-10-20 15:27:57] CHECKPOINT | 4,338 plants saved


...

[2026-10-20 19:23:17] CHECKPOINT | 5,838 plants saved


 99%|█████████▉| 1799/1815 [4:42:24<02:41, 10.07s/it]

[2026-10-20 19:39:01] CHECKPOINT | 5,938 plants saved


100%|██████████| 1815/1815 [4:44:59<00:00,  9.42s/it]


[2026-10-20 19:41:34] CHECKPOINT | 5,953 plants saved
[2026-10-20 19:41:34] Extracted 17,484 headers
[2026-10-20 19:41:35] Saved: wfo_descriptions_with_headers.csv
[2026-10-20 19:41:35] Matched 17,484 / 23,489 rows


In [ ]:
retry_strategy = Retry(
    total=3,
    connect=3,
    read=3,
    backoff_factor=2,
    status_forcelist=[
        429,
        500,
        502,
        503,
        504
    ],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(
    max_retries=retry_strategy
)

session = requests.Session()

session.mount("http://", adapter)
session.mount("https://", adapter)

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/130.0 Safari/537.36"
    )
})

df = pd.read_csv(INPUT_FILE)

def log(message):
    timestamp = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )
    print(f"[{timestamp}] {message}")

log(f"Loaded {len(df):,} plants")

processed_ids = set()

if os.path.exists(OUTPUT_FILE):

    try:

        existing = pd.read_csv(
            OUTPUT_FILE,
            usecols=["ipni_id"]
        )

        processed_ids = set(
            existing["ipni_id"]
            .astype(str)
            .unique()
        )

        log(
            f"Found existing output. "
            f"Skipping "
            f"{len(processed_ids):,} "
            f"already processed plants."
        )

    except Exception as e:

        log(
            f"Could not load existing "
            f"checkpoint: {e}"
        )

records_buffer = []

success_count = len(processed_ids)
failure_count = 0

start_time = time.time()

def plants_per_minute():

    elapsed_minutes = (
        time.time() - start_time
    ) / 60

    if elapsed_minutes == 0:
        return 0

    return success_count / elapsed_minutes


def save_batch():

    global records_buffer

    if not records_buffer:
        return

    batch_df = pd.DataFrame(
        records_buffer
    )

    file_exists = os.path.exists(
        OUTPUT_FILE
    )

    batch_df.to_csv(
        OUTPUT_FILE,
        mode="a",
        header=not file_exists,
        index=False
    )

    log(
        f"SAVED "
        f"{len(batch_df):,} rows "
        f"to {OUTPUT_FILE}"
    )

    records_buffer = []

for plant_num, row in enumerate(
    tqdm(
        df.itertuples(index=False),
        total=len(df)
    ),
    start=1
):

    ipni_id = str(row.ipni_id)

    if ipni_id in processed_ids:
        continue

    url = (
        f"https://www.worldfloraonline.org/"
        f"ipni/{ipni_id}"
    )

    try:

        response = session.get(
            url,
            timeout=30,
            verify=False
        )

        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        citation_lookup = {}

        bibliography = soup.find(
            "section",
            id="bibliography"
        )

        if bibliography:

            for li in bibliography.select(
                "ul > li"
            ):

                anchor = li.find(
                    "a",
                    id=True
                )

                if anchor:

                    citation_lookup[
                        anchor["id"]
                    ] = li.get_text(
                        " ",
                        strip=True
                    )

        general = soup.find(
            "div",
            class_="tab-pane active"
        )

        found_description = False

        general = soup.find(
            "div",
            class_="tab-pane active"
        )

        if general:

            descriptions = general.find_all(
                "div",
                class_="description-with-citations"
            )

            for desc_idx, desc in enumerate(
                descriptions
            ):

                p = desc.find("p")

                description_text = (
                    p.get_text(
                        " ",
                        strip=True
                    )
                    if p
                    else None
                )

                if description_text:
                    found_description = True

                provider = desc.find(
                    "li",
                    style=lambda s:
                    s and "float: right" in s
                )

                provider_text = (
                    provider.get_text(
                        " ",
                        strip=True
                    )
                    if provider
                    else None
                )

                citation_ids = []
                citation_texts = []

                citations_ul = desc.find(
                    "ul",
                    class_="citations"
                )

                if citations_ul:

                    for a in citations_ul.find_all(
                        "a"
                    ):

                        href = a.get("href")

                        if (
                            href
                            and href.startswith("#")
                        ):

                            citation_id = href[1:]

                            citation_ids.append(
                                citation_id
                            )

                            if (
                                citation_id
                                in citation_lookup
                            ):
                                citation_texts.append(
                                    citation_lookup[
                                        citation_id
                                    ]
                                )

                records_buffer.append({
                    "ipni_id": ipni_id,
                    "description_id": desc_idx,
                    "description": description_text,
                    "provider": provider_text,
                    "citation_ids":
                        ";".join(citation_ids),
                    "citations":
                        " || ".join(citation_texts),
                    "url": url
                })

        if not found_description:

            records_buffer.append({
                "ipni_id": ipni_id,
                "description_id": None,
                "description": None,
                "provider": None,
                "citation_ids": None,
                "citations": None,
                "url": url
            })

        success_count += 1
        processed_ids.add(ipni_id)

        if success_count % 25 == 0:

            log(
                f"PROGRESS | "
                f"success={success_count:,} | "
                f"failed={failure_count:,} | "
                f"rate={plants_per_minute():.1f} "
                f"plants/min"
            )

        if success_count % BATCH_SIZE == 0:

            save_batch()

            log(
                f"CHECKPOINT | "
                f"{success_count:,}/"
                f"{len(df):,} plants"
            )

    except Exception as e:

        failure_count += 1

        log(
            f"ERROR | "
            f"{ipni_id} | {e}"
        )

        pd.DataFrame(
            [{
                "ipni_id": ipni_id,
                "error": str(e)
            }]
        ).to_csv(
            FAILED_FILE,
            mode="a",
            header=not os.path.exists(
                FAILED_FILE
            ),
            index=False
        )

    time.sleep(SLEEP_TIME)

save_batch()

elapsed = (
    time.time() - start_time
) / 60

log("FINISHED")
log(f"Success: {success_count:,}")
log(f"Failed : {failure_count:,}")
log(f"Elapsed : {elapsed:.1f} min")
log(
    f"Rate    : "
    f"{plants_per_minute():.1f} plants/min"
)